# Task 05: Phone Detection + Alert System

The COMPLETE ExamGuard v0.1 phone detector:
- Detects phones via webcam
- Plays beep sound on detection
- Saves screenshot evidence with timestamp
- Has cooldown to prevent alert spam

**Press 'q' to quit.**

## Cell 1: Setup + Configuration

All settings in ONE place. Change these numbers to tune the system.

In [ ]:
from ultralytics import YOLO
import cv2
import time
import os
from datetime import datetime
import winsound  # Windows beep — only works on Windows

# =============================================
# CONFIGURATION — Change these to tune behavior
# =============================================
CONFIDENCE_THRESHOLD = 0.5    # Minimum confidence to count as phone (0.0 to 1.0)
ALERT_COOLDOWN = 5            # Seconds between alerts (prevents beeping every frame)
BEEP_FREQUENCY = 1000         # Beep sound frequency in Hz (higher = higher pitch)
BEEP_DURATION = 300           # Beep length in milliseconds
EVIDENCE_FOLDER = 'evidence'  # Folder to save detection screenshots

# Create evidence folder if it doesn't exist
os.makedirs(EVIDENCE_FOLDER, exist_ok=True)

# Load YOLO
model = YOLO('yolov8n.pt')

# Find phone class ID
PHONE_CLASS_ID = None
for cid, name in model.names.items():
    if 'phone' in name.lower():
        PHONE_CLASS_ID = cid
        break

print("ExamGuard v0.1 — Phone Detection System")
print("=" * 45)
print(f"  Phone class ID:    {PHONE_CLASS_ID}")
print(f"  Confidence:        {CONFIDENCE_THRESHOLD*100:.0f}%")
print(f"  Alert cooldown:    {ALERT_COOLDOWN} seconds")
print(f"  Evidence folder:   {EVIDENCE_FOLDER}/")
print("=" * 45)
print("Ready!")

## Cell 2: Run ExamGuard v0.1

**The COMPLETE system.** This combines everything from Tasks 02-04 plus alerts.

**What you'll see:**
- Green status bar = no phone detected (all clear)
- Red status bar + red box + BEEP = phone detected!
- Screenshot auto-saved to `evidence/` folder

**Press 'q' to quit.**

In [ ]:
camera = cv2.VideoCapture(0)

if not camera.isOpened():
    print("ERROR: Cannot open webcam!")
else:
    print("ExamGuard v0.1 RUNNING!")
    print("Place a phone on the desk to test.")
    print("Press 'q' to quit.\n")
    
    frame_count = 0
    start_time = time.time()
    last_alert_time = 0       # When was the last alert?
    total_detections = 0      # Total phone detections this session
    screenshots_saved = 0     # How many screenshots saved
    
    while True:
        success, frame = camera.read()
        if not success:
            break
        
        # Get frame dimensions for drawing
        height, width = frame.shape[:2]
        
        # Run YOLO
        results = model(frame, verbose=False)
        
        # Check each detection
        phone_detected = False
        phone_confidence = 0
        
        for box in results[0].boxes:
            class_id = int(box.cls[0])
            confidence = float(box.conf[0])
            
            # Only phones above threshold
            if class_id == PHONE_CLASS_ID and confidence >= CONFIDENCE_THRESHOLD:
                phone_detected = True
                phone_confidence = max(phone_confidence, confidence)
                total_detections += 1
                
                # Draw red box around phone
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 3)
                label = f"PHONE {confidence*100:.0f}%"
                cv2.putText(frame, label, (x1, y1 - 10),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
        
        # Draw status bar at top
        current_time = time.time()
        
        if phone_detected:
            # RED status bar
            cv2.rectangle(frame, (0, 0), (width, 45), (0, 0, 200), -1)  # -1 = filled
            cv2.putText(frame, f"PHONE DETECTED! ({phone_confidence*100:.0f}%)",
                       (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 255), 2)
            
            # Alert with cooldown
            if current_time - last_alert_time >= ALERT_COOLDOWN:
                last_alert_time = current_time
                
                # BEEP!
                winsound.Beep(BEEP_FREQUENCY, BEEP_DURATION)
                
                # Save screenshot evidence
                timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
                filename = f"{EVIDENCE_FOLDER}/phone_{timestamp}.jpg"
                cv2.imwrite(filename, frame)
                screenshots_saved += 1
                print(f"  ALERT! Phone detected ({phone_confidence*100:.0f}%) — Screenshot saved: {filename}")
        else:
            # GREEN status bar
            cv2.rectangle(frame, (0, 0), (width, 45), (0, 120, 0), -1)
            cv2.putText(frame, "No phones detected — All clear",
                       (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 255), 2)
        
        # FPS counter
        frame_count += 1
        elapsed = current_time - start_time
        fps = frame_count / elapsed if elapsed > 0 else 0
        cv2.putText(frame, f"FPS: {fps:.1f} | Detections: {total_detections} | Screenshots: {screenshots_saved}",
                   (10, height - 15), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (200, 200, 200), 1)
        
        # Show frame
        cv2.imshow('ExamGuard v0.1 — Phone Detection (Press q to quit)', frame)
        
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
    
    camera.release()
    cv2.destroyAllWindows()
    
    # Session summary
    print("\n" + "=" * 45)
    print("SESSION SUMMARY")
    print("=" * 45)
    print(f"  Duration:       {elapsed:.0f} seconds")
    print(f"  Frames:         {frame_count}")
    print(f"  Average FPS:    {fps:.1f}")
    print(f"  Phone alerts:   {screenshots_saved}")
    print(f"  Evidence saved:  {EVIDENCE_FOLDER}/")
    print("=" * 45)

## Cell 3: Check saved evidence

Let's see what screenshots were saved.

In [ ]:
import matplotlib.pyplot as plt
import glob

# Find all saved screenshots
screenshots = sorted(glob.glob(f"{EVIDENCE_FOLDER}/phone_*.jpg"))

if not screenshots:
    print("No screenshots saved yet. Run Cell 2 with a phone visible to the camera.")
else:
    print(f"Found {len(screenshots)} evidence screenshots:\n")
    
    # Show up to 4 latest screenshots
    show = screenshots[-4:]  # Last 4
    fig, axes = plt.subplots(1, len(show), figsize=(5*len(show), 4))
    if len(show) == 1:
        axes = [axes]
    
    for ax, path in zip(axes, show):
        img = cv2.imread(path)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        ax.imshow(img_rgb)
        ax.set_title(os.path.basename(path), fontsize=8)
        ax.axis('off')
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nAll screenshots saved in: {os.path.abspath(EVIDENCE_FOLDER)}/")

## ExamGuard v0.1 COMPLETE!

### What you built:
- Real-time phone detection via webcam
- Filtered to phones only (ignores people, chairs, etc.)
- Visual alert (red bar + red box)
- Sound alert (beep)
- Evidence screenshots saved with timestamps
- Configurable threshold and cooldown

### What you learned:
- How YOLO works (pre-trained, 80 objects, bounding boxes)
- How webcam video loops work (capture → detect → draw → display → repeat)
- How to filter detections (class ID + confidence threshold)
- How to build an alert system (cooldown, evidence, sound)
- The threshold tradeoff (strict vs lenient)

### Next module: 02_gaze_tracking
Detect where students are looking — are they looking at their own paper or at their neighbor's?